In [1]:
import pandas as pd
# Assume you have a DataFrame named 'df'
# with a column 'Category_Column' and another 'Value_Column'
data=pd.read_csv('C:/Users/Yap Zheng Xian/Documents/Programming/Extension/notebook/Dataset/Cleaned Dataset.csv')

data.columns

Index(['input', 'label'], dtype='object')

In [2]:
data['label'].unique()

array(['phishing', 'legitimate'], dtype=object)

In [ ]:
# Define the condition for the rows
condition = data['label'] == 'legitimate'

# Define the specific columns you want to extract
desired_columns = ['input']

# Extract the data using .loc[]
legitimate_website= data.loc[condition, desired_columns]

legitimate_website.to_csv('cookies.csv',index=False)

In [5]:
legitimate_website.count()

input    5911191
dtype: int64

# Import Selenium to WebScarping

# CookiesSearch.ORG

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

def scrape_cookie_data(search_term):
    # Setup Chrome options (headless mode is optional but recommended for scraping)
    chrome_options = Options()
    # chrome_options.add_argument("--headless")  # Uncomment to run without opening a window
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    # Initialize the driver
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)

    try:
        # 1. Navigate to the website
        url = "https://cookiesearch.org/"
        print(f"Navigating to {url}...")
        driver.get(url)

        # 2. Find the search input and search
        # We wait for an input field that is likely the search bar.
        # Adjust the selector below if the specific ID is known (e.g., By.ID, "search-box")
        wait = WebDriverWait(driver, 10)
        
        print(f"Searching for: {search_term}")
        search_box = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "input[type='text'], input[type='search']")))
        search_box.clear()
        search_box.send_keys(search_term)
        search_box.send_keys(Keys.RETURN)

        # 3. Wait for results and click the first result
        print("Waiting for results...")
        time.sleep(1000)
        
        # We wait for a link (<a> tag) to appear that is NOT the nav links. 
        # Usually, results appear in a main container. We try to click the first meaningful link.
        # Note: You may need to inspect the site to see if results are in a specific table or div.
        # Here we assume the result is a link containing the search term or inside a result table.
        
        # Strategy: Wait for any link that is clickable and likely part of the content
        # This selector looks for a link inside a table or list, or simply the first link in the main body
        first_result = wait.until(EC.element_to_be_clickable((By.XPATH, "(//a[contains(@href, '/cookie/')])[1]")))
        
        result_url = first_result.get_attribute('href')
        print(f"Clicking result: {result_url}")
        first_result.click()

        # 4. Extract the <div class="cookie-details"> part
        print("Waiting for details page...")
        
        # The user specified <div="cookie-details"> which implies class="cookie-details"
        # We wait for this specific element to be visible
        cookie_details_div = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "cookie-details")))
        
        # Extract text and inner HTML
        details_text = cookie_details_div.text
        details_html = cookie_details_div.get_attribute('outerHTML')

        print("\n--- Extracted Data (Text) ---")
        print(details_text)
        
        # Optional: Save to file
        # with open("cookie_data.html", "w", encoding="utf-8") as f:
        #     f.write(details_html)

    except Exception as e:
        print(f"An error occurred: {e}")
        # If the specific class isn't found, print the page source to debug
        # print(driver.page_source)

    finally:
        print("\nClosing driver...")
        driver.quit()

if __name__ == "__main__":
    # Example: Searching for the common Google Analytics cookie "_ga"
    scrape_cookie_data("cookieyes-consent")


# Cookies Serve

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import csv
from bs4 import BeautifulSoup
import pandas as pd

def get_detailed_report(html):
    soup = BeautifulSoup(html, 'html.parser')
    
    # Find the element with the specific ID
    report_section = soup.find(id="detailed-report")
    #print("Report Section:",report_section)
    
    if report_section:
        # Return the HTML string of the section
        return report_section.prettify()
    else:
        return "Detailed report section not found."

def extract_to_csv(html, filename="cookie_report.csv"):
    soup = BeautifulSoup(html, 'html.parser')
    
    # Locate the table body and rows
    report_section = soup.find(id="detailed-report")
    report_section.prettify()
    #print("Report Section: ",report_section)
    # The structure is div.report-tbody -> div.report-trow
    rows = soup.select('.report-tbody .report-trow')
    #rows = soup.select('.report-tbody .report-trow')
    
    extracted_data = []
    
    for row in rows:
        # Find all columns within the row
        cols = row.find_all(class_='report-tcol')
        
        if len(cols) == 5:
            # Extract text and strip whitespace
            cookie_name = cols[0].get_text(strip=True)
            domain = cols[1].get_text(strip=True)
            description = cols[2].get_text(strip=True)
            duration = cols[3].get_text(strip=True)
            cookie_type = cols[4].get_text(strip=True)
            
            # You requested the order: cookies, domain, description, type, duration
            # Note: In the HTML, Duration is index 3 and Type is index 4.
            # We swap them here to match your requested CSV format.
            extracted_data.append([cookie_name, domain, description, cookie_type])

    # Write to CSV
    with open(filename, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        # Write Header
        writer.writerow(['Cookies', 'Domain', 'Description', 'Type'])
        # Write Data
        writer.writerows(extracted_data)
        
    print(f"Successfully saved {len(extracted_data)} rows to {filename}")


def rows_have_text(d):
  rows = d.find_elements(By.CSS_SELECTOR, ".report-tbody .report-trow")
  return any(row.text.strip() for row in rows)

def scrape_cookie_data(search_term):
    # Setup Chrome options (headless mode is optional but recommended for scraping)
    chrome_options = Options()
    chrome_options.add_argument("--headless")  # Uncomment to run without opening a window
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    # Initialize the driver
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)

    try:
        # 1. Navigate to the website
        url = "https://www.cookieserve.com/"
        #print(f"Navigating to {url}...")
        driver.get(url)

        # 2. Find the search input and search
        # We wait for an input field that is likely the search bar.
        # Adjust the selector below if the specific ID is known (e.g., By.ID, "search-box")
        wait = WebDriverWait(driver, 10)
        
        print(f"Searching for: {search_term}")
        search_box = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "input[type='text'], input[type='search']")))
        search_box.clear()
        search_box.send_keys(search_term)
        search_box.send_keys(Keys.RETURN)

        # 3. Wait for results and click the first result
        print("Waiting for results...")
        time.sleep(3)
        driver.execute_script("window.scrollBy(0, 1000);")
        time.sleep(5)
        '''
        WebDriverWait(driver, 20).until
        (
            EC.presence_of_element_located((By.CLASS_NAME, ".report-tbody .report-trow"))
        )
        '''
        WebDriverWait(driver, 25).until(rows_have_text)
        
        #time.sleep(15)
        #result_box = wait.until(EC.element_to_be_clickable((By.CLASS_NAME,"report-trow")))
        html= driver.page_source
        #print("HTML:",html)
        output=get_detailed_report(html)
        #print("Output:",output)
        extract_to_csv(output)
        
    except Exception as e:
        print(f"An error occurred: {e}")
        # If the specific class isn't found, print the page source to debug
        # print(driver.page_source)

    finally:
        print("\nClosing driver...")
        driver.quit()


if __name__ == "__main__":
  data=pd.read_csv('C:/Users/Yap Zheng Xian/Documents/Programming/Extension/notebook/legitimatewebsite.csv')
  for item in data['input']:
    scrape_cookie_data(item)


# Scraping HTML Structure
Beautiful Soup Documentation:https://www.crummy.com/software/BeautifulSoup/bs4/doc/

In [1]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import csv
from bs4 import BeautifulSoup
import pandas as pd

#to be modify
def extract_html_to_csv(html, filename="phishingwebsite.csv"):
    soup = BeautifulSoup(html, 'html.parser')
    
    # Locate the table body and rows
    report_section = soup.find(id="detailed-report")
    report_section.prettify()
    #print("Report Section: ",report_section)
    # The structure is div.report-tbody -> div.report-trow
    rows = soup.select('.report-tbody .report-trow')
    #rows = soup.select('.report-tbody .report-trow')
    
    extracted_data = []
    
    for row in rows:
        # Find all columns within the row
        cols = row.find_all(class_='report-tcol')
        
        if len(cols) == 5:
            # Extract text and strip whitespace
            cookie_name = cols[0].get_text(strip=True)
            domain = cols[1].get_text(strip=True)
            description = cols[2].get_text(strip=True)
            duration = cols[3].get_text(strip=True)
            cookie_type = cols[4].get_text(strip=True)
            
            # You requested the order: cookies, domain, description, type, duration
            # Note: In the HTML, Duration is index 3 and Type is index 4.
            # We swap them here to match your requested CSV format.
            extracted_data.append([cookie_name, domain, description, cookie_type])

    # Write to CSV
    with open(filename, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        # Write Header
        writer.writerow(['HTML Structure', 'Description', 'Type'])
        # Write Data
        writer.writerows(extracted_data)
        
    print(f"Successfully saved {len(extracted_data)} rows to {filename}")

def clean_html_tag(html):
    soup = BeautifulSoup(html, 'html.parser')
    
    # Find the element with the specific ID
    #html_section = soup.find(id="detailed-report")
    text = soup.find_all('a')
    print("<a> element ",text)
    
    li= soup.find_all('li')
    print("<li> element ",li)
    
    link= soup.find_all('link')
    print("<link> element ",link)
    
    '''
    for link in soup.find_all('a'):
       text=""
       text=text +"\n" +link.get('href')
    #text=soup.get_text()
    '''
    if text | li | link:
        # Return the HTML string of the section
        return text+ li+ link
        #return text.prettify()
    else:
        return "HTML not found."

def scrape_website_structure(url):
    # Setup Chrome options (headless mode is optional but recommended for scraping)
    chrome_options = Options()
    chrome_options.add_argument("--headless")  # Uncomment to run without opening a window
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    # Initialize the driver
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)

    try:
        # 1. Navigate to the website
        driver.get(url)

        # 2. Find the search input and search
        # We wait for an input field that is likely the search bar.
        # Adjust the selector below if the specific ID is known (e.g., By.ID, "search-box")
        wait = WebDriverWait(driver, 10)
        
        print(f"Scraping Website for {url}")

        time.sleep(8)
        
        #time.sleep(15)
        html= driver.page_source
        #print("HTML:",html)
        output=clean_html_tag(html)
        print("Output:",output)
        #extract_html_to_csv(output)
        
    except Exception as e:
        print(f"An error occurred: {e}")
        # If the specific class isn't found, print the page source to debug
        # print(driver.page_source)

    finally:
        print("\nClosing driver...")
        driver.quit()

if __name__ == "__main__":
  #data=pd.read_csv('C:/Users/Yap Zheng Xian/Documents/Programming/Extension/notebook/legitimatewebsite.csv')
  #for item in data['input']:
    url="https://cookiesearch.org/"
    scrape_website_structure(url)


Scraping Website for https://cookiesearch.org/
<a> element  [<a href="https://www.cookieyes.com/product/cookie-consent/?ref=cypbcyb&amp;utm_source=cookie-banner&amp;utm_medium=powered-by-cookieyes" rel="noopener" style="margin-left:5px;line-height:0" target="_blank"><img alt="Cookieyes logo" src="https://cdn-cookieyes.com/assets/images/poweredbtcky.svg" style="width:78px;height:13px;margin:0"/></a>, <a href="https://cookiesearch.org" title="Cookie Search">
<img alt="Cookie Search" height="27" src="https://cookiesearch.org/wp-content/themes/cookie-search/assets/images/cookie-search-logo.svg" width="223"/>
</a>, <a href="#">About</a>, <a href="https://cookiesearch.org/cookies/?cookie-cat=necessary" title="View cookies">View cookies</a>, <a href="https://cookiesearch.org/cookies/?cookie-cat=analytics" title="View cookies">View cookies</a>, <a href="https://cookiesearch.org/cookies/?cookie-cat=advertisement" title="View cookies">View cookies</a>, <a href="https://cookiesearch.org/cookies/?

# Unused Code

In [2]:
from bs4 import BeautifulSoup

# Paste your full HTML string here
html_content = """
<body class="wp-singular page-template...">
    ... (your full HTML content) ...
</body>
"""

def get_detailed_report(html):
    soup = BeautifulSoup(html, 'html.parser')
    
    # Find the element with the specific ID
    report_section = soup.find(id="detailed-report")
    print("Report Section:",report_section)
    
    if report_section:
        # Return the HTML string of the section
        return "Report Section after Prettify by Beautiful Soup"+report_section.prettify()
    else:
        return "Detailed report section not found."

if __name__ == "__main__":
    # Assuming you have the HTML in a file named 'source.html'
    # You can read it like this:
    # with open("source.html", "r", encoding="utf-8") as f:
    #     html_content = f.read()

    result = get_detailed_report(html_content)
    print(result)
    
    # Optionally save it to a new file
    with open("detailed_report_only.html", "w", encoding="utf-8") as f:
        f.write(result)


Report Section: None
Detailed report section not found.


In [ ]:
# The HTML content provided in your request
'''
html_content = """
<div id="detailed-report" bis_skin_checked="1">
    <h3>Detailed cookie scan report for&nbsp;<span class="scan-domain">google.com</span></h3>
    <p class="cs-scan-text">Cookieserve scans only the homepage of the website. To check cookies on all pages, try our <a href="https://www.cookieyes.com/cookie-scanner/?ref=CS" class="cy-advanced-scanner">advanced cookie scanner</a> for free.</p>
    <div class="report-table" bis_skin_checked="1">
        <div class="report-thead" bis_skin_checked="1">
            <div class="report-tcol" bis_skin_checked="1">Cookie</div>
            <div class="report-tcol" bis_skin_checked="1">Domain</div>
            <div class="report-tcol" bis_skin_checked="1">Description</div>
            <div class="report-tcol" bis_skin_checked="1">Duration</div>
            <div class="report-tcol" bis_skin_checked="1">Type</div>
        </div>
        <div class="report-tbody" bis_skin_checked="1"><div class="report-trow" bis_skin_checked="1"><div class="report-tcol" bis_skin_checked="1">AEC</div><div class="report-tcol" bis_skin_checked="1">.google.com</div><div class="report-tcol" bis_skin_checked="1">AEC cookies are used to ensure user data remains consistent during an Analytics session.</div><div class="report-tcol" bis_skin_checked="1">6 months</div><div class="report-tcol" bis_skin_checked="1">Analytics</div></div><div class="report-trow" bis_skin_checked="1"><div class="report-tcol" bis_skin_checked="1">__Secure-ENID</div><div class="report-tcol" bis_skin_checked="1">.google.com</div><div class="report-tcol" bis_skin_checked="1">The __Secure-ENID cookie is a type of secure cookie used for authentication and to ensure the security of user sessions.</div><div class="report-tcol" bis_skin_checked="1">1 year 1 month</div><div class="report-tcol" bis_skin_checked="1">Necessary</div></div><div class="report-trow" bis_skin_checked="1"><div class="report-tcol" bis_skin_checked="1">__Secure-BUCKET</div><div class="report-tcol" bis_skin_checked="1">.google.com</div><div class="report-tcol" bis_skin_checked="1"></div><div class="report-tcol" bis_skin_checked="1">6 months</div><div class="report-tcol" bis_skin_checked="1">Other</div></div><div class="report-trow" bis_skin_checked="1"><div class="report-tcol" bis_skin_checked="1">sb_wiz.zpc.gws-wiz.</div><div class="report-tcol" bis_skin_checked="1">google.com</div><div class="report-tcol" bis_skin_checked="1"></div><div class="report-tcol" bis_skin_checked="1">never</div><div class="report-tcol" bis_skin_checked="1">Other</div></div><div class="report-trow" bis_skin_checked="1"><div class="report-tcol" bis_skin_checked="1">hsb;;1770017382639</div><div class="report-tcol" bis_skin_checked="1">google.com</div><div class="report-tcol" bis_skin_checked="1"></div><div class="report-tcol" bis_skin_checked="1">session</div><div class="report-tcol" bis_skin_checked="1">Other</div></div><div class="report-trow" bis_skin_checked="1"><div class="report-tcol" bis_skin_checked="1">hsb;;1770017382638</div><div class="report-tcol" bis_skin_checked="1">google.com</div><div class="report-tcol" bis_skin_checked="1"></div><div class="report-tcol" bis_skin_checked="1">session</div><div class="report-tcol" bis_skin_checked="1">Other</div></div><div class="report-trow" bis_skin_checked="1"><div class="report-tcol" bis_skin_checked="1">_c;;i</div><div class="report-tcol" bis_skin_checked="1">google.com</div><div class="report-tcol" bis_skin_checked="1"></div><div class="report-tcol" bis_skin_checked="1">session</div><div class="report-tcol" bis_skin_checked="1">Other</div></div><div class="report-trow" bis_skin_checked="1"><div class="report-tcol" bis_skin_checked="1">TESTCOOKIESENABLED</div><div class="report-tcol" bis_skin_checked="1">.google.com</div><div class="report-tcol" bis_skin_checked="1"></div><div class="report-tcol" bis_skin_checked="1">session</div><div class="report-tcol" bis_skin_checked="1">Other</div></div></div>
    </div>
</div>
"""
'''

def extract_to_csv(html, filename="cookie_report.csv"):
    soup = BeautifulSoup(html, 'html.parser')
    
    # Locate the table body and rows
    report_section = soup.find(id="detailed-report")
    report_section.prettify()
    print("Report Section: ",report_section)
    # The structure is div.report-tbody -> div.report-trow
    rows = soup.select('.report-tbody .report-trow')
    #rows = soup.select('.report-tbody .report-trow')
    
    extracted_data = []
    
    for row in rows:
        # Find all columns within the row
        cols = row.find_all(class_='report-tcol')
        
        if len(cols) == 5:
            # Extract text and strip whitespace
            cookie_name = cols[0].get_text(strip=True)
            domain = cols[1].get_text(strip=True)
            description = cols[2].get_text(strip=True)
            duration = cols[3].get_text(strip=True)
            cookie_type = cols[4].get_text(strip=True)
            
            # You requested the order: cookies, domain, description, type, duration
            # Note: In the HTML, Duration is index 3 and Type is index 4.
            # We swap them here to match your requested CSV format.
            extracted_data.append([cookie_name, domain, description, cookie_type, duration])

    # Write to CSV
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        # Write Header
        writer.writerow(['Cookies', 'Domain', 'Description', 'Type', 'Duration'])
        # Write Data
        writer.writerows(extracted_data)
        
    print(f"Successfully saved {len(extracted_data)} rows to {filename}")

if __name__ == "__main__":
    extract_to_csv(html_content)


In [ ]:
if __name__ == "__main__":
    # Example: Searching for the common Google Analytics cookie "_ga"
    scrape_cookie_data("google.com")

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Configure options for headless browsing
opts = Options()
opts.add_argument("--headless=new")
opts.add_argument("--window-size=1920,1080")

# Initialize the driver
driver = webdriver.Chrome(options=opts)

try:
    # Navigate to the target page
    driver.get("https://news.ycombinator.com/") # Example URL

    # Wait for content to ensure the page is loaded
    WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.CLASS_NAME, "athing"))
    )

    # Get the full page source after JavaScript execution
    full_html_source = driver.page_source

    print("Scraped page source snippet (first 500 characters):")
    print(full_html_source[:500])


    # Extract specific data (e.g., article titles)
    stories = driver.find_elements(By.CLASS_NAME, "athing")
    print(f"\nFound {len(stories)} stories.")
    for story in stories[:3]:
        title_link = story.find_element(By.CSS_SELECTOR, "span.titleline > a")
        print(f"- {title_link.text}: {title_link.get_attribute('href')}")
        
    # 2. Scraping CSS File Links
    # CSS is usually found in <link> tags with rel="stylesheet"
    css_links = driver.find_elements(By.TAG_NAME, "link")
    print("--- CSS FILES ---")
    for link in css_links:
        rel = link.get_attribute("rel")
        if rel == "stylesheet":
            href = link.get_attribute("href")
            print(f"Found CSS: {href}")

    # 3. Scraping JavaScript File Links
    # JS is found in <script> tags with a "src" attribute
    js_scripts = driver.find_elements(By.TAG_NAME, "script")
    print("\n--- JAVASCRIPT FILES ---")
    for script in js_scripts:
        src = script.get_attribute("src")
        if src:  # Only print if it's an external file, not inline code
            print(f"Found JS: {src}")


finally:
    # Always close the browser
    driver.quit()


In [ ]:
import os
import requests
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
import jsbeautifier

# Setup folders
BASE_DIR = "Dataset/website_dataset"
os.makedirs(f"{BASE_DIR}/css", exist_ok=True)
os.makedirs(f"{BASE_DIR}/js", exist_ok=True)

opts = Options()
opts.add_argument("--headless=new")
driver = webdriver.Chrome(options=opts)

def download_file(url, folder):
    if not url or not url.startswith("http"):
        return
    
    # Create a filename from the end of the URL
    filename = url.split("/")[-1].split("?")[0] # Remove version queries like ?v=1.2
    if not filename:
        return

    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            with open(f"{BASE_DIR}/{folder}/{filename}", "w", encoding="utf-8") as f:
                f.write(response.text)
            print(f"Successfully saved: {filename}")
    except Exception as e:
        print(f"Failed to download {url}: {e}")

try:
    driver.get("https://google.com/") # Replace with your target
    WebDriverWait(driver, 15).until(EC.visibility_of_element_located((By.TAG_NAME, "body")))
    
    final_html = driver.execute_script("return document.body.innerHTML")
    with open("structure.html", "w", encoding="utf-8") as f:
      f.write(final_html)

    # 1. Download CSS
    links = driver.find_elements(By.TAG_NAME, "link")
    for link in links:
        if link.get_attribute("rel") == "stylesheet":
            download_file(link.get_attribute("href"), "css")

    # 2. Download JS
    scripts = driver.find_elements(By.TAG_NAME, "script")
    for script in scripts:
        src = script.get_attribute("src")
        src = jsbeautifier.beautify(src)
        if src:
            download_file(src, "js")

finally:
    driver.quit()
    print("\nDownload complete. Check the 'website_dataset' folder.")